In [10]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [13]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
275,Tyra Banks needs to teach these girls that it'...,negative
8,This movie is silly and very short of being a ...,negative
135,This started out as a good sketch comedy. The ...,negative
588,ok we have a film that some are calling one of...,negative
24,We're a long way from LAURA. Once again Otto P...,positive


In [16]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [19]:
df = normalize_text(df)
df.head()

,review,sentiment
275,tyra bank need teach girl beautiful outside in...,negative
8,movie silly short funny movie unhappy easterne...,negative
135,started good sketch comedy first show good loo...,negative
588,ok film calling one best movie ever but sittin...,negative
24,long way laura otto preminger directs dana and...,positive


In [22]:
df['sentiment'].value_counts()

sentiment
negative    251
positive    249
Name: count, dtype: int64

In [25]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [28]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
275,tyra bank need teach girl beautiful outside in...,0
8,movie silly short funny movie unhappy easterne...,0
135,started good sketch comedy first show good loo...,0
588,ok film calling one best movie ever but sittin...,0
24,long way laura otto preminger directs dana and...,1


In [31]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [34]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [40]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/dushyantkv508/MLOps-Capstone-Project.mlflow')
dagshub.init(repo_owner='dushyantkv508', repo_name='MLOps-Capstone-Project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

C:\Users\Shubham\anaconda3\envs\atlas\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6b46ac38-a01f-432d-801e-dd1917a5d960&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b5d3a4e93b43c0d8ca1d5ada1958888c7d89a7db6ee0516281b14d47b73ba5c2




Accessing as dushyantkv508

Initialized MLflow to track repo "dushyantkv508/MLOps-Capstone-Project"

Repository dushyantkv508/MLOps-Capstone-Project initialized!

2026/06/01 14:02:48 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4958637887d844d893fd94c358dc894c', creation_time=1780302767961, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780302767961, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [43]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-06-01 14:03:31,695 - INFO - Starting MLflow run...
2026-06-01 14:03:33,165 - INFO - Logging preprocessing parameters...
2026-06-01 14:03:34,217 - INFO - Initializing Logistic Regression model...
2026-06-01 14:03:34,218 - INFO - Fitting the model...
2026-06-01 14:03:34,273 - INFO - Model training complete.
2026-06-01 14:03:34,275 - INFO - Logging model parameters...
2026-06-01 14:03:34,601 - INFO - Making predictions...
2026-06-01 14:03:34,602 - INFO - Calculating evaluation metrics...
2026-06-01 14:03:34,621 - INFO - Logging evaluation metrics...
2026-06-01 14:03:36,047 - INFO - Saving and logging the model...
2026/06/01 14:03:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 14:03:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run nosy-bee-848 at: https://dagshub.com/dushyantkv508/MLOps-Capstone-Project.mlflow/#/experiments/0/runs/2a3124b613e54b39a21c3dc70f6507c3
🧪 View experiment at: https://dagshub.com/dushyantkv508/MLOps-Capstone-Project.mlflow/#/experiments/0
